In [0]:
# 1. load daily traffic 
from pyspark.sql import functions as F

traffic_df = spark.table("workspace.redfin_gold.daily_traffic")

historical_df = (
    traffic_df
    .filter(
        (F.col("is_missing_date") == False) &
        (F.col("is_future_date") == False)
    )
)

print("Historical date range:")
historical_df.agg(
    F.min("day").alias("start_date"),
    F.max("day").alias("end_date"),
    F.count("*").alias("observed_days"),
    F.sum("total_sessions").alias("total_sessions")
).show()

In [0]:
print(traffic_df.count())
traffic_df.show(5)

In [0]:
# 2. filter observed historical dates
historical_df = traffic_df.filter(
    (F.col("is_missing_date") == False) &
    (F.col("is_future_date") == False)
)

print("Observed historical rows:", historical_df.count())

historical_df.agg(
    F.min("day").alias("start_date"),
    F.max("day").alias("end_date"),
    F.sum("total_sessions").alias("total_sessions")
).show()

In [0]:
# 3. seasonal index 
monthly_2018 = (
    historical_df
    .filter(F.col("year") == 2018)
    .groupBy("month")
    .agg(
        F.count("*").alias("observed_days"),
        F.sum("total_sessions").alias("monthly_sessions"),
        F.avg("total_sessions").alias("avg_daily_sessions")
    )
    .orderBy("month")
)

display(monthly_2018)

In [0]:
# calc the 2018 annual average daily sessions, excluding incomplete september
# sep =( aug+oct )/2 

annual_avg_2018 = (
    monthly_2018
    .filter(F.col("month") != 9)
    .agg(
        (
            F.sum("monthly_sessions") /
            F.sum("observed_days")
        ).alias("annual_avg_daily_sessions")
    )
)

# calc seasonal indices for the 10 complete months

seasonal_indices = (
    monthly_2018
    .filter(F.col("month") != 9)
    .crossJoin(annual_avg_2018)
    .withColumn(
        "seasonal_index",
        F.col("avg_daily_sessions") /
        F.col("annual_avg_daily_sessions")
    )
    .select(
        "month",
        "observed_days",
        "avg_daily_sessions",
        "annual_avg_daily_sessions",
        "seasonal_index"
    )
    .orderBy("month")
)

display(seasonal_indices)

In [0]:
september_index = (
    seasonal_indices
    .agg(
        (
            F.max(F.when(F.col("month") == 8, F.col("seasonal_index"))) +
            F.max(F.when(F.col("month") == 10, F.col("seasonal_index")))
        ).__truediv__(2).alias("seasonal_index")
    )
    .select(
        F.lit(9).alias("month"),
        "seasonal_index"
    )
)

# combine  all month indices with interpolated september

seasonal_factors = (
    seasonal_indices
    .select("month", "seasonal_index")
    .unionByName(september_index)
    .orderBy("month")
)

display(seasonal_factors)

In [0]:
## 4 calc 2019 monthly average daily sessions
monthly_2019 = (
    historical_df
    .filter(F.col("year") == 2019)
    .groupBy("year_month", "month")
    .agg(
        F.count("*").alias("observed_days"),
        F.avg("total_sessions").alias("avg_daily_sessions")
    )
)

# remove the historical seasonal effect to see if traffic is generally growthing or bc of seasonality 
# assuming 2018 seasonaliy is applicable to 2019 
adjusted_2019 = (
    monthly_2019
    .join(seasonal_factors, on="month", how="inner")
    .withColumn(
        "seasonally_adjusted_sessions",
        F.col("avg_daily_sessions") / F.col("seasonal_index")
    )
    .select(
        "year_month",
        "observed_days",
        F.round("avg_daily_sessions", 0).alias("avg_daily_sessions"),
        F.round("seasonal_index", 3).alias("seasonal_index"),
        F.round("seasonally_adjusted_sessions", 0)
         .alias("seasonally_adjusted_sessions")
    )
    .orderBy("year_month")
)

display(adjusted_2019)

In [0]:
# 5. underlying trend 
from pyspark.sql.window import Window

trend_window = Window.orderBy("year_month")

traffic_trend = (
    adjusted_2019
    .withColumn(
        "previous_month_sessions",
        F.lag("seasonally_adjusted_sessions").over(trend_window)
    )
    .withColumn(
        "monthly_change",
        F.col("seasonally_adjusted_sessions") -
        F.col("previous_month_sessions")
    )
    .withColumn(
        "monthly_change_pct",
        F.col("monthly_change") /
        F.col("previous_month_sessions") * 100
    )
    .select(
        "year_month",
        "seasonally_adjusted_sessions",
        F.round("monthly_change", 0).alias("monthly_change"),
        F.round("monthly_change_pct", 1).alias("monthly_change_pct")
    )
    .orderBy("year_month")
)

display(traffic_trend)

In [0]:
# 6. forecast 
from pyspark.sql import functions as F

trend_anchors = (
    adjusted_2019
    .agg(
        F.max(
            F.when(
                F.col("year_month") == "2019-01",
                F.col("seasonally_adjusted_sessions")
            )
        ).alias("january"),
        F.max(
            F.when(
                F.col("year_month") == "2019-05",
                F.col("seasonally_adjusted_sessions")
            )
        ).alias("may"),
        F.max(
            F.when(
                F.col("year_month") == "2019-07",
                F.col("seasonally_adjusted_sessions")
            )
        ).alias("july")
    )
    .first()
)

january = trend_anchors["january"]
may = trend_anchors["may"]
july = trend_anchors["july"]

upside_rate = (may / january) ** (1 / 4) - 1
downside_rate = (july / may) ** (1 / 2) - 1

print(f"January adjusted daily sessions: {january:,.0f}")
print(f"May adjusted daily sessions: {may:,.0f}")
print(f"July adjusted daily sessions: {july:,.0f}")
print(f"Upside monthly growth reference: {upside_rate:.2%}")
print(f"Downside monthly decline reference: {downside_rate:.2%}")

In [0]:
#7. aug-dec forecast 
forecast_months = [
    ("2019-08", 8),
    ("2019-09", 9),
    ("2019-10", 10),
    ("2019-11", 11),
    ("2019-12", 12)
]

scenario_rates = {
    "upside": upside_rate,
    "baseline": 0.0,
    "downside": downside_rate
}

forecast_rows = []

for scenario, rate in scenario_rates.items():
    previous_level = july

    for year_month, month in forecast_months:
        forecast_level = previous_level * (1 + rate)

        forecast_rows.append((
            scenario,
            year_month,
            month,
            rate,
            forecast_level
        ))

        previous_level = forecast_level

underlying_forecast = spark.createDataFrame(
    forecast_rows,
    [
        "scenario",
        "year_month",
        "month",
        "monthly_rate",
        "underlying_daily_sessions"
    ]
)

display(
    underlying_forecast
    .withColumn(
        "underlying_daily_sessions",
        F.round("underlying_daily_sessions", 0)
    )
    .orderBy("scenario", "month")
)

Databricks visualization. Run in Databricks to view.

In [0]:
# apply seasonality 
forecast_with_seasonality = (
    underlying_forecast
    .join(
        seasonal_factors.select("month", "seasonal_index"),
        on="month",
        how="left"
    )
    .withColumn(
        "forecast_avg_daily_sessions",
        F.col("underlying_daily_sessions") * F.col("seasonal_index")
    )
    .select(
        "scenario",
        "year_month",
        "month",
        "underlying_daily_sessions",
        "seasonal_index",
        "forecast_avg_daily_sessions"
    )
    .orderBy("scenario", "month")
)

display(
    forecast_with_seasonality
    .withColumn(
        "forecast_avg_daily_sessions",
        F.round("forecast_avg_daily_sessions", 0)
    )
)

Databricks visualization. Run in Databricks to view.

In [0]:
# monthly total now 
forecast_monthly = (
    forecast_with_seasonality
    .withColumn(
        "forecast_date",
        F.to_date(F.concat(F.col("year_month"), F.lit("-01")))
    )
    .withColumn(
        "calendar_days",
        F.dayofmonth(F.last_day("forecast_date"))
    )
    .withColumn(
        "forecast_monthly_sessions",
        F.col("forecast_avg_daily_sessions") * F.col("calendar_days")
    )
    .select(
        "scenario",
        "year_month",
        "calendar_days",
        F.round("forecast_avg_daily_sessions", 0)
            .alias("forecast_avg_daily_sessions"),
        F.round("forecast_monthly_sessions", 0)
            .alias("forecast_monthly_sessions")
    )
    .orderBy("scenario", "year_month")
)

display(forecast_monthly)

Databricks visualization. Run in Databricks to view.

In [0]:
scenario_summary = (
    forecast_monthly
    .groupBy("scenario")
    .agg(
        F.sum("forecast_monthly_sessions").alias("total_forecast_sessions"),
        F.sum("calendar_days").alias("total_calendar_days")
    )
    .withColumn(
        "avg_daily_sessions",
        F.col("total_forecast_sessions") / F.col("total_calendar_days")
    )
    .select(
        "scenario",
        F.round("total_forecast_sessions", 0).alias("total_forecast_sessions"),
        F.round("avg_daily_sessions", 0).alias("avg_daily_sessions")
    )
    .orderBy("scenario")
)

display(scenario_summary)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# actual daily sessions from january to july 2019
actual_pd = (
    monthly_2019
    .select("year_month", "avg_daily_sessions")
    .orderBy("year_month")
    .toPandas()
)

# forecast daily sessions from august to december 2019
forecast_pd = (
    forecast_with_seasonality
    .select(
        "scenario",
        "year_month",
        "forecast_avg_daily_sessions"
    )
    .orderBy("year_month")
    .toPandas()
)

# prepare chart
plt.figure(figsize=(13, 6))

# actual traffic
actual_y = actual_pd["avg_daily_sessions"] / 1_000_000
plt.plot(
    actual_pd["year_month"],
    actual_y,
    marker="o",
    linewidth=2.5,
    label="Actual"
)

# add data labels for actual traffic
for x, y in zip(actual_pd["year_month"], actual_y):
    plt.annotate(
        f"{y:.1f}M",
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=9,
        fontweight="bold"
    )

# vertical offsets per scenario to prevent overlapping labels
scenario_offsets = {
    "upside": 8,
    "baseline": 6,
    "downside": -14
}

# forecast scenarios
for scenario in ["upside", "baseline", "downside"]:
    scenario_data = forecast_pd[
        forecast_pd["scenario"] == scenario
    ]
    forecast_y = scenario_data["forecast_avg_daily_sessions"] / 1_000_000
    
    plt.plot(
        scenario_data["year_month"],
        forecast_y,
        marker="o",
        linewidth=2,
        linestyle="--",
        label=scenario.capitalize()
    )
    
    # add data labels for forecast scenarios
    for x, y in zip(scenario_data["year_month"], forecast_y):
        plt.annotate(
            f"{y:.1f}M",
            (x, y),
            textcoords="offset points",
            xytext=(0, scenario_offsets[scenario]),
            ha="center",
            fontsize=8.5
        )

# separate actual and forecast periods
plt.axvline(
    x=6.5,
    color="gray",
    linestyle=":",
    linewidth=1.5
)

# forecast annotation
plt.text(
    9.5,
    2.60,
    "Forecast scenarios",
    ha="center",
    fontsize=11,
    color="gray"
)

# chart formatting
months = [
    "2019-01", "2019-02", "2019-03", "2019-04",
    "2019-05", "2019-06", "2019-07", "2019-08",
    "2019-09", "2019-10", "2019-11", "2019-12"
]

month_labels = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

plt.xticks(months, month_labels)
plt.title(
    "Redfin Traffic: Actual vs. Forecast (2019)",
    fontsize=15
)
plt.xlabel("Month")
plt.ylabel("Average Daily Sessions (Millions)")
plt.legend()
plt.grid(axis="y", alpha=0.3)

# add slight padding at the top/bottom so edge labels don't get cut off
ymin, ymax = plt.ylim()
plt.ylim(ymin - 0.1, ymax + 0.15)

plt.tight_layout()
plt.show()


In [0]:

import matplotlib.pyplot as plt
import pandas as pd

# actual daily sessions from january to july 2019
actual_pd = (
    monthly_2019
    .select("year_month", "avg_daily_sessions")
    .orderBy("year_month")
    .toPandas()
)

# forecast daily sessions from august to december 2019
forecast_pd = (
    forecast_with_seasonality
    .select(
        "scenario",
        "year_month",
        "forecast_avg_daily_sessions"
    )
    .orderBy("year_month")
    .toPandas()
)

# calculate calendar days in each month
actual_pd["days_in_month"] = (
    pd.PeriodIndex(actual_pd["year_month"], freq="M").days_in_month
)

forecast_pd["days_in_month"] = (
    pd.PeriodIndex(forecast_pd["year_month"], freq="M").days_in_month
)

# calculate total monthly sessions
actual_pd["total_monthly_sessions"] = (
    actual_pd["avg_daily_sessions"] * actual_pd["days_in_month"]
)

forecast_pd["forecast_total_monthly_sessions"] = (
    forecast_pd["forecast_avg_daily_sessions"] * forecast_pd["days_in_month"]
)

# prepare chart
plt.figure(figsize=(13, 6))

# actual monthly traffic
actual_y = actual_pd["total_monthly_sessions"] / 1_000_000

plt.plot(
    actual_pd["year_month"],
    actual_y,
    marker="o",
    linewidth=2.5,
    label="Actual"
)

for x, y in zip(actual_pd["year_month"], actual_y):
    plt.annotate(
        f"{y:.1f}M",
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=9,
        fontweight="bold"
    )

# forecast scenarios
scenario_offsets = {
    "upside": 8,
    "baseline": 6,
    "downside": -14
}

for scenario in ["upside", "baseline", "downside"]:
    scenario_data = forecast_pd[
        forecast_pd["scenario"] == scenario
    ]

    forecast_y = (
        scenario_data["forecast_total_monthly_sessions"] / 1_000_000
    )

    plt.plot(
        scenario_data["year_month"],
        forecast_y,
        marker="o",
        linewidth=2,
        linestyle="--",
        label=scenario.capitalize()
    )

    for x, y in zip(scenario_data["year_month"], forecast_y):
        plt.annotate(
            f"{y:.1f}M",
            (x, y),
            textcoords="offset points",
            xytext=(0, scenario_offsets[scenario]),
            ha="center",
            fontsize=8.5
        )

# separate actual and forecast periods
plt.axvline(
    x=6.5,
    color="gray",
    linestyle=":",
    linewidth=1.5
)

# forecast annotation
plt.text(
    9.5,
    plt.ylim()[1] * 0.97,
    "Forecast scenarios",
    ha="center",
    fontsize=11,
    color="gray"
)

# chart formatting
months = [
    "2019-01", "2019-02", "2019-03", "2019-04",
    "2019-05", "2019-06", "2019-07", "2019-08",
    "2019-09", "2019-10", "2019-11", "2019-12"
]

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.xticks(months, month_labels)
plt.title(
    "Redfin Traffic: Actual vs. Forecast (2019)",
    fontsize=15
)
plt.xlabel("Month")
plt.ylabel("Total Monthly Sessions (Millions)")
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


In [0]:
# compare 2018 vs 2019 forecast 
forecast_yoy = (
    forecast_with_seasonality
    .join(
        monthly_2018.select(
            "month",
            F.col("avg_daily_sessions").alias("2018_avg_daily_sessions"),
            F.col("observed_days").alias("2018_observed_days")
        ),
        on="month",
        how="left"
    )
    .withColumn(
        "yoy_growth_pct",
        F.when(
            F.col("2018_observed_days") >= 28,
            (
                F.col("forecast_avg_daily_sessions")
                / F.col("2018_avg_daily_sessions") - 1
            ) * 100
        )
    )
    .select(
        "scenario",
        "year_month",
        F.round("forecast_avg_daily_sessions", 0)
            .alias("forecast_avg_daily_sessions"),
        F.round("2018_avg_daily_sessions", 0)
            .alias("2018_avg_daily_sessions"),
        F.round("yoy_growth_pct", 1).alias("yoy_growth_pct")
    )
    .orderBy("scenario", "year_month")
)

display(forecast_yoy)